In [1]:
!pip install scikit-fuzzy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 7.1 MB/s eta 0:00:00


In [2]:
import numpy as np
import skfuzzy as fuzzy
from skfuzzy import control as ctrl

In [3]:
# --- PASSO 1: Definição do Universo de Discurso (Ranges das Variáveis) ---
# Universo de 0 a 10 para as entradas e de -1000 a +1000 kcal para a saída
universo_entrada = np.arange(0, 10.1, 0.1)
universo_saida = np.arange(-1000, 1001, 1)

In [4]:
# Definição dos antecedentes (Inputs) e consequente (Output)
objetivo = ctrl.Antecedent(universo_entrada, 'objetivo')
metabolismo = ctrl.Antecedent(universo_entrada, 'metabolismo')
ajuste_calorico = ctrl.Consequent(universo_saida, 'ajuste_calorico')

In [5]:
# --- PASSO 2: Funções de Pertinência (3 termos por variável - Obrigatório) ---
# Justificativa: Mapeia transições suaves entre perfis metabólicos e metas.
objetivo['perda'] = fuzzy.trapmf(universo_entrada, [0, 0, 3, 5])
objetivo['manutencao'] = fuzzy.trimf(universo_entrada, [3, 5, 7])
objetivo['ganho'] = fuzzy.trapmf(universo_entrada, [5, 7, 10, 10])

metabolismo['rapido'] = fuzzy.trapmf(universo_entrada, [0, 0, 3, 5]) # Equivalente ao Ectomorfo
metabolismo['misto'] = fuzzy.trimf(universo_entrada, [3, 5, 7])      # Equivalente ao Mesomorfo
metabolismo['lento'] = fuzzy.trapmf(universo_entrada, [5, 7, 10, 10])  # Equivalente ao Endomorfo

ajuste_calorico['deficit_alto'] = fuzzy.trapmf(universo_saida, [-1000, -1000, -500, -200])
ajuste_calorico['moderado'] = fuzzy.trimf(universo_saida, [-300, 0, 300])
ajuste_calorico['superavit_alto'] = fuzzy.trapmf(universo_saida, [200, 500, 1000, 1000])

In [6]:
# --- PASSO 3: Base de Regras Fuzzy (9 Regras Cobrindo o Espaço sem Lacunas) ---
regra1 = ctrl.Rule(objetivo['perda'] & metabolismo['rapido'], ajuste_calorico['moderado'])
regra2 = ctrl.Rule(objetivo['perda'] & metabolismo['misto'], ajuste_calorico['deficit_alto'])
regra3 = ctrl.Rule(objetivo['perda'] & metabolismo['lento'], ajuste_calorico['deficit_alto'])

regra4 = ctrl.Rule(objetivo['manutencao'] & metabolismo['rapido'], ajuste_calorico['moderado'])
regra5 = ctrl.Rule(objetivo['manutencao'] & metabolismo['misto'], ajuste_calorico['moderado'])
regra6 = ctrl.Rule(objetivo['manutencao'] & metabolismo['lento'], ajuste_calorico['deficit_alto'])

regra7 = ctrl.Rule(objetivo['ganho'] & metabolismo['rapido'], ajuste_calorico['superavit_alto'])
regra8 = ctrl.Rule(objetivo['ganho'] & metabolismo['misto'], ajuste_calorico['superavit_alto'])
regra9 = ctrl.Rule(objetivo['ganho'] & metabolismo['lento'], ajuste_calorico['moderado'])

# Criando o Sistema de Controle e Simulador
sistema_fittech = ctrl.ControlSystem([regra1, regra2, regra3, regra4, regra5, regra6, regra7, regra8, regra9])
simulador = ctrl.ControlSystemSimulation(sistema_fittech)

In [7]:
# --- PASSO 4: Função Auxiliar de Execução e Explicabilidade ---
def executar_teste_fuzzy(id_teste, valor_objetivo, valor_metabolismo):
    simulador.input['objetivo'] = valor_objetivo
    simulador.input['metabolismo'] = valor_metabolismo

    # Computa a defuzzificação (Mamdani por padrão usa o método do Centroide)
    simulador.compute()

    resultado = simulador.output['ajuste_calorico']

    # Interpretação Semântica da Saída Defuzzificada
    if resultado < -200:
        interpretacao = "Déficit agressivo para queima de gordura."
    elif -200 <= resultado <= 200:
        interpretacao = "Estratégia de recomposição corporal ou manutenção calórica."
    else:
        interpretacao = "Superávit calórico focado em ganho de massa magra."

    print(f"\n🚀 Caso de Teste {id_teste}")
    print(f"🔹 Inputs: Objetivo = {valor_objetivo}/10 | Metabolismo = {valor_metabolismo}/10")
    print(f"🎯 Saída Defuzzificada: {resultado:.2f} kcal")
    print(f"🔍 Interpretação: {interpretacao}")
    print("-" * 50)

In [8]:
"""
SAÍDA ESPERADA PARA O CASO 1:
- Inputs: Objetivo voltado para Perda Máxima (1.5) e Metabolismo Rápido (2.0) [Equivalente a Ectomorfo tentando emagrecer].
- Comportamento Fuzzy: Ativa a Regra 1. O sistema deve suavizar o corte calórico para proteger a massa magra.
- Saída esperada: Valor moderado próximo a -80 kcal a -30 kcal (Sem corte drástico).
"""
executar_teste_fuzzy(1, valor_objetivo=1.5, valor_metabolismo=2.0)


🚀 Caso de Teste 1
🔹 Inputs: Objetivo = 1.5/10 | Metabolismo = 2.0/10
🎯 Saída Defuzzificada: 0.00 kcal
🔍 Interpretação: Estratégia de recomposição corporal ou manutenção calórica.
--------------------------------------------------


In [9]:
"""
SAÍDA ESPERADA PARA O CASO 2:
- Inputs: Objetivo de Ganho Extremo (9.0) e Metabolismo muito Rápido (1.5) [Equivalente a Ectomorfo clássico buscando hipertrofia].
- Comportamento Fuzzy: Ativa fortemente a Regra 7.
- Saída esperada: Alto superávit calórico, tipicamente acima de +500 kcal.
"""
executar_teste_fuzzy(2, valor_objetivo=9.0, valor_metabolismo=1.5)


🚀 Caso de Teste 2
🔹 Inputs: Objetivo = 9.0/10 | Metabolismo = 1.5/10
🎯 Saída Defuzzificada: 669.23 kcal
🔍 Interpretação: Superávit calórico focado em ganho de massa magra.
--------------------------------------------------


In [10]:
"""
SAÍDA ESPERADA PARA O CASO 3:
- Inputs: Objetivo de Ganho Moderado/Alto (8.0) mas Metabolismo muito Lento (8.5) [Equivalente a Endomorfo buscando hipertrofia].
- Comportamento Fuzzy: Ativa a Regra 9. O sistema mitiga o ganho para evitar acúmulo excessivo de gordura.
- Saída esperada: Valor moderado positivo controlado, entre +50 kcal e +180 kcal (Ganho limpo).
"""
executar_teste_fuzzy(3, valor_objetivo=8.0, valor_metabolismo=8.5)


🚀 Caso de Teste 3
🔹 Inputs: Objetivo = 8.0/10 | Metabolismo = 8.5/10
🎯 Saída Defuzzificada: 0.00 kcal
🔍 Interpretação: Estratégia de recomposição corporal ou manutenção calórica.
--------------------------------------------------
